# Enveda CASMI 2026 — Offline Kaggle Inference & Submission Notebook

**Competition Compliance Checklist:**
- **Internet Access:** Disabled (`Offline` execution mode).
- **Runtime:** `< 15 minutes` on Kaggle CPU or GPU (well under the `<= 9 hours` limit).
- **Attached Inputs:**
  1. Competition Data: `/kaggle/input/enveda-casmi-2026` (`test.parquet`)
  2. Trained Model Dataset (exported from Colab): `/kaggle/input/casmi26-trained-models` (contains `casmi26/` package, `candidate_pool.parquet`, and `casmi_pipeline_models.pkl`)
  3. *(Optional)* Candidate Shards: `/kaggle/input/casmi-2026-candidate-database`
- **Output:** `/kaggle/working/submission.csv` (`molecule_id`, `smiles` with up to 25 semicolon-separated, `inchikey14`-deduplicated SMILES per molecule).

In [ ]:
import sys
import time
from pathlib import Path
import pandas as pd

t0 = time.time()

# 1. Locate attached model dataset & competition data (supports /kaggle/input, /kaggle/inputs, & local verification)
MODEL_SEARCH_PATHS = [
    Path('/kaggle/input/casmi26-trained-models'),
    Path('/kaggle/inputs/casmi26-trained-models'),
    Path('/content/drive/MyDrive/CASMI2026/kaggle_dataset_export'),
    Path('/mnt/LMSTUDIO/CASMI/kaggle_dataset_export'),
    Path('kaggle_dataset_export'),
]
MODEL_DIR = next((p for p in MODEL_SEARCH_PATHS if (p / 'casmi_pipeline_models.pkl').exists()), MODEL_SEARCH_PATHS[-1])
for p in [MODEL_DIR, Path('/mnt/LMSTUDIO/CASMI'), Path('.')]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from casmi26.models import CASMIPipeline

print(f'[INFO] Loading trained CASMI 2026 pipeline from: {MODEL_DIR}')
pipeline = CASMIPipeline.load_checkpoint(MODEL_DIR)
print(f'[OK] Loaded CandidatePool with {len(pipeline.candidate_pool):,} unique inchikey14 structures.')

In [ ]:
# 2. Load test.parquet (automatically swapped by Kaggle with the ~400-molecule hidden test set during scoring)
TEST_SEARCH_PATHS = [
    Path('/kaggle/input/enveda-casmi-2026/test.parquet'),
    Path('/kaggle/inputs/enveda-casmi-2026/test.parquet'),
    Path('/content/drive/MyDrive/CASMI2026/data/test.parquet'),
    Path('/mnt/LMSTUDIO/CASMI/data/test.parquet'),
    Path('data/test.parquet'),
    MODEL_DIR / 'synthetic_data' / 'test_synthetic.parquet',
]
found_test = next((p for p in TEST_SEARCH_PATHS if p.exists()), None)

if found_test is not None:
    print(f'[INFO] Reading test spectra from: {found_test}')
    test_df = pd.read_parquet(found_test)
    out_csv = Path('/kaggle/working/submission.csv') if Path('/kaggle/working').exists() else Path('submission.csv')
else:
    from tests.test_pipeline import generate_synthetic_casmi26_data
    from casmi26.validation import sample_timstof_like_validation_panel
    synth_df, _ = generate_synthetic_casmi26_data()
    test_df, _ = sample_timstof_like_validation_panel(synth_df, n_molecules=25)
    out_csv = Path('/mnt/LMSTUDIO/CASMI/submission.csv')

n_spectra = len(test_df)
n_molecules = test_df['molecule_id'].nunique()
print(f'[INFO] Loaded test.parquet: {n_spectra:,} spectra across {n_molecules:,} unique molecule_ids.')

In [ ]:
# 3. Run Multi-Spectrum Aggregation, Candidate Retrieval, Neural Fingerprint + GBDT Reranking
submission_df = pipeline.predict_top25_for_test_df(test_df, max_k=25)

# 4. Strict Submission Schema & Rule Validation
assert list(submission_df.columns) == ['molecule_id', 'smiles'], f'Invalid columns: {submission_df.columns}'
assert len(submission_df) == n_molecules, f'Expected {n_molecules} rows, got {len(submission_df)}'
assert submission_df['molecule_id'].nunique() == n_molecules, 'Duplicate molecule_ids found!'
assert submission_df['smiles'].isna().sum() == 0, 'Null SMILES predictions found!'
assert (submission_df['smiles'].str.len() > 0).all(), 'Empty SMILES string found!'

submission_df.to_csv(out_csv, index=False)
elapsed = time.time() - t0
print(f'[SUCCESS] Wrote valid submission.csv to {out_csv} in {elapsed:.2f}s')
submission_df.head()